# Frontier benchmark — reproduce a classification run (§14.11 protocol)

**What this reproduces:** one canonical frontier-benchmark run — the 9-method, 5-fold, seed-42
protocol of master-report §14.11 (TabPFN + CatBoost + LightGBM + XGBoost + RF + LR + 3 GLMs,
scored on log loss / AUC / Brier / PR-AUC / top-decile lift, with parameter counts and the
D3 beyond-SE Pareto rule).

**How it works:** this notebook does *not* reimplement the experiment. It calls the exact
script that produced the committed results — `scripts/eval/insurance_benchmark_v1/run_frontier_benchmark.py` —
so the code run here is byte-for-byte the code that ran the research.

**Requirements:**
- Kernel must be the benchmark venv (`.venv-ta`, Python 3.12, tabpfn-client 0.3.3) — see `notebooks/reproducibility/README.md`
- `TABPFN_API_KEY` set in the environment or in the repo-root `.env` (scripts fail with a clear error if missing)

**Cost:** this notebook runs ONE dataset (`coil2000`, 9,822 rows — the smallest). A few minutes.
The full 6-dataset suite takes hours (TabPFN is a hosted API; the 163K-row sets dominate).

**⚠ Warning:** seed 42 is the *canonical* seed. Re-running it **overwrites the committed**
`frontier_results_coil2000.csv` / `frontier_plot_coil2000.png`. For exploration use `--seed 7`
(writes `_seed7` files, never clobbers canonical results — see the last cell).

In [ ]:
from pathlib import Path
import sys

# Locate the repo root regardless of where Jupyter was launched from.
ROOT = next(p for p in [Path.cwd(), *[Path.cwd().parents[i] for i in range(1, 4)]]
            if (p / "scripts/eval/insurance_benchmark_v1/run_frontier_benchmark.py").exists())
print("repo root:", ROOT)

In [ ]:
# Preflight: versions + API key presence (prints only whether the key exists, never the key).
import importlib, os
for m in ("tabpfn_client", "pandas", "sklearn", "xgboost", "lightgbm", "catboost"):
    mod = importlib.import_module(m)
    print(f"{m:14s} {getattr(mod, '__version__', '?')}")
key = bool(os.environ.get("TABPFN_API_KEY")) or ((ROOT / ".env").exists() and bool((ROOT / ".env").read_text()))
print("API key present:", key)
assert key, "TABPFN_API_KEY missing — set it in the environment or repo-root .env"

In [ ]:
# The exact command that produced the canonical coil2000 frontier run (master report §14.11).
# --pr-auc adds PR AUC + top-decile lift to the base log-loss/AUC/Brier protocol.
import subprocess, time

cmd = [sys.executable, "scripts/eval/insurance_benchmark_v1/run_frontier_benchmark.py", "--pr-auc", "coil2000"]
print("running:", " ".join(cmd))
t0 = time.time()
r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-4000:])
print(f"exit: {r.returncode}  ({time.time() - t0:.0f}s)")
if r.returncode:
    print(r.stderr[-2000:])

In [ ]:
# The evidence this run produced — same schema as the committed canonical file:
# method, mean, se, mean_auc, se_auc, mean_brier, se_brier, n_params, on_frontier
import pandas as pd
CSV = ROOT / "scripts/eval/insurance_benchmark_v1/frontier_results_coil2000.csv"
df = pd.read_csv(CSV)
df.round(4)

In [ ]:
# The frontier plot (x = log10(n_params), y = mean log loss ± SE; red = on frontier).
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
img = mpimg.imread(ROOT / "scripts/eval/insurance_benchmark_v1/frontier_plot_coil2000.png")
plt.figure(figsize=(11, 6))
plt.imshow(img); plt.axis("off"); plt.show()

## How to extend this to the full suite

- **All 6 classification datasets:** drop the dataset argument —
  `python scripts/eval/insurance_benchmark_v1/run_frontier_benchmark.py --pr-auc` (hours; do it overnight).
- **Regression/frequency axis (§14.8):** `--regression` — same protocol, RMSE / Poisson deviance, 5-fold KFold.
- **Without clobbering canonical results:** `--seed 7` writes `frontier_results_*_seed7.csv`. Seed-stability
  testing is part of the protocol (§14.12 used seeds 7/42/123).

**Reading the table:** `on_frontier=yes` = not dominated under the D3 beyond-SE rule
(dominated iff some method with strictly fewer params has `mean_B + SE_B < mean_A − SE_A`).
On this dataset TabPFN should be log-loss best with the GLM family also on the frontier —
the master report's most robust result (S11/S12 of the learning path).

**Version honesty:** every number here is stamped to `model_path="v3_default"` + tabpfn-client 0.3.3.
If either drifts, master-report §15 requires a re-test before trusting the verdict — see also the
sweep-reuse caveat (the frontier reuses `home_turf_sweep_results.csv` rows on 3 datasets).